# S&P 500 CEO Name Extraction
### Parsing and Cleaning Key People Data from the S&P 500 Index

---

**Overview:**  
This notebook loads a dataset of S&P 500 companies enriched with "Key People" information scraped from Wikipedia. Because the raw data contains noisy pipe-delimited strings with bracketed citation numbers, we:

1. Load and inspect the dataset
2. Explore and understand the messy `Key People` field
3. Prototype a cleaning utility to strip bracket artifacts
4. Build a regex-based extractor to isolate the CEO name for each company
5. Validate the results on the first five entries

**Dataset:** `sp500_with_key_people.csv`  
**Source:** Google Drive / Colab Notebooks

## 1. Setup & Imports

Import all required libraries and mount Google Drive to access the dataset.

In [20]:
import pandas as pd       # Data manipulation and analysis
import numpy as np        # Numerical operations
import matplotlib.pyplot as plt  # Plotting (available for future visualizations)
import seaborn as sns     # Statistical data visualization
import re                 # Regular expressions for text parsing

# Mount Google Drive to access shared datasets
from google.colab import drive
drive.mount('/content/drive')

## 2. Load the Dataset

Read the S&P 500 CSV file from Google Drive. Robust error handling is included to catch missing files or other I/O issues.

In [22]:
file_path = '/content/drive/My Drive/Colab Notebooks/sp500_with_key_people.csv'

try:
    df = pd.read_csv(file_path)
    print('CSV loaded successfully. First 5 rows:')
    df.head(5)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the file exists and the path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

CSV loaded successfully. First 5 rows:


## 3. Initial Data Inspection

Preview the first few rows to understand the structure of the dataset.  
Note the `Key People` column — it contains pipe-delimited (`|`) strings with citation references like `[ 1 ]` that will need to be cleaned.

In [23]:
# Display the first 5 rows to get a feel for the data
df.head()

  Symbol             Security              GICSSector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry    Headquarters Location  Date added  \
0        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04   
1               Building Products     Milwaukee, Wisconsin  2017-07-26   
2           Health Care Equipment  North Chicago, Illinois  1957-03-04   
3                   Biotechnology  North Chicago, Illinois  2012-12-31   
4  IT Consulting & Other Services          Dublin, Ireland  2011-07-06   

       CIK      Founded                                         Key People  
0    66740         1902  William M. Brown | (chairman and CEO) | [ | 3 | ]  
1    91142         1916  Kevin J. Wheeler | [ | 2 | ] | ( 

## 4. Exploring the Raw `Key People` Field

Before building a full parser, let's look at the raw `Key People` strings after a simple pipe removal.  
This reveals the citation bracket artifacts (e.g., `[  3  ]`) that clutter the names and titles.

In [25]:
# Quick preview: remove pipe delimiters to read the raw Key People strings
# We can already see citation brackets like '[  3  ]' that need to be stripped
for entry in list(df["Key People"].head(5)):
    print(entry.replace("|", ""))

William M. Brown  (chairman and CEO)  [  3  ]
Kevin J. Wheeler  [  2  ]  (  chairman  ,  president  &  CEO  )
Robert B. Ford  (  chairman  &  CEO  )  [  1  ]  Robert Funck  (  EVP  &  CFO  )
Robert Michael (  chairman and CEO  )
Julie Sweet  (  chair  and  CEO  )


## 5. Prototype: Removing Bracketed Citation Numbers

Before applying regex to the full column, we prototype a helper function on a small list.  
This function filters out `[`, `]`, and any digit tokens that were part of a citation (e.g., `[ 2 ]`).

> **Note:** This token-based approach works on pre-split lists. In the final solution (Section 6), we use `re.sub` directly on the raw string, which is cleaner and more robust.

In [24]:
# Sample tokenized Key People entries for prototyping the cleaning logic
data_lists = [
    ['Kevin J. Wheeler', '[', '2', ']', '(', 'chairman', ',', 'president', '&', 'CEO', ')'],
    ['Robert B. Ford', '(', 'chairman', '&', 'CEO', ')', '[', '1', ']', 'Robert Funck', '(', 'EVP', '&', 'CFO', ')']
]

def remove_bracketed_numbers(input_list):
    """
    Remove citation bracket tokens from a tokenized Key People list.
    Filters out '[', ']', and any digit token that appears between them.
    
    Args:
        input_list (list): Tokenized list of name/title components.
    
    Returns:
        list: Cleaned list with citation artifacts removed.
    """
    return [
        item for item in input_list
        if not (
            item == '[' or
            item == ']' or
            (
                item.isdigit() and
                input_list[input_list.index(item) - 1] == '[' and
                input_list[input_list.index(item) + 1] == ']'
            )
        )
    ]

# Apply and display cleaned results
cleaned_lists = [remove_bracketed_numbers(lst) for lst in data_lists]
for cleaned_list in cleaned_lists:
    print(cleaned_list)

['Kevin J. Wheeler', '(', 'chairman', ',', 'president', '&', 'CEO', ')']
['Robert B. Ford', '(', 'chairman', '&', 'CEO', ')', 'Robert Funck', '(', 'EVP', '&', 'CFO', ')']


## 6. CEO Name Extraction

Now we apply the full extraction pipeline to the entire dataset.

**Strategy:**
1. Remove all `|` pipe delimiters
2. Strip bracketed citation numbers using `re.sub` (e.g., `[ 1 ]` → `''`)
3. Use a regex pattern to find a name followed by a parenthetical role containing **"CEO"** or **"Chief Executive Officer"**
4. Return `None` if no match is found

The result is stored in a new `CEO Name` column on the dataframe.

In [26]:
def extract_ceo_name(key_people_str):
    """
    Extract the CEO's name from a raw 'Key People' string.

    The raw strings are pipe-delimited and contain Wikipedia citation brackets.
    This function:
      - Removes '|' characters
      - Strips bracketed citation numbers like '[ 1 ]'
      - Uses regex to find a name followed by a role containing 'CEO'

    Args:
        key_people_str (str): Raw value from the 'Key People' column.

    Returns:
        str or None: The extracted CEO name, or None if not found.
    """
    if not isinstance(key_people_str, str):
        return None

    # Step 1: Remove pipe '|' delimiters left over from Wikipedia scraping
    cleaned_str = key_people_str.replace("|", "")

    # Step 2: Remove bracketed citation numbers, e.g. '[ 1 ]', '[ 23 ]'
    cleaned_str = re.sub(r'\[\s*\d+\s*\]', '', cleaned_str)

    # Step 3: Match a name (letters, spaces, dots, hyphens) followed by
    # a parenthetical that contains 'CEO' or 'Chief Executive Officer'
    match = re.search(
        r'([A-Za-z\s\.-]+?)\s*\((?:[^()]*?(?:CEO|Chief Executive Officer)[^()]*?)\)',
        cleaned_str,
        re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    # Return None if no CEO entry could be identified
    return None


# Apply the extractor to the entire 'Key People' column
df['CEO Name'] = df['Key People'].apply(extract_ceo_name)

# Display the Security name and extracted CEO for the first 5 companies
print("CEOs for the first 5 companies:")
display(df[['Security', 'CEO Name']].head(5))

CEOs for the first 5 companies:


              Security          CEO Name
0                   3M  William M. Brown
1          A. O. Smith  Kevin J. Wheeler
2  Abbott Laboratories    Robert B. Ford
3               AbbVie    Robert Michael
4            Accenture       Julie Sweet